# Étape 1 — Collecte des données : chargement de la base `HuileData2003.mdb`

**Objectif de cette étape :** ouvrir la base de données Access, voir ce qu'elle contient,
puis charger dans Python **uniquement les tables utiles** à notre étude sur les transformateurs de puissance.

**Méthode choisie :**
- la base est un fichier **Microsoft Access** (`.mdb`) le fichier HuileData2003(1).mdb;
- sous Windows, le moyen le plus direct et le plus fiable de la lire est le **pilote ODBC Microsoft Access** ;
- en Python, la bibliothèque **`pyodbc`** se connecte à ce pilote et **`pandas`** transforme chaque table en tableau (DataFrame) ;
- on ouvre la base en **lecture seule** : le fichier d'origine n'est jamais modifié ;
- à la fin, on enregistre une **copie CSV** de chaque table : les étapes suivantes pourront travailler sans Access.

Aucun nettoyage n'est fait ici : on charge les données **telles qu'elles sont**.

## 1. Importer les bibliothèques

In [1]:
# pathlib : pour manipuler les chemins de fichiers (dossiers, noms de fichiers)
from pathlib import Path

# unicodedata : pour enlever les accents des noms de fichiers CSV (ex. "Équipement" -> "Equipement")
import unicodedata

# warnings : pour masquer un message d'avertissement de pandas (expliqué plus bas)
import warnings

# pandas : la bibliothèque qui gère les tableaux de données (on l'appelle "pd" par convention)
import pandas as pd

# pyodbc : la bibliothèque qui permet à Python de "parler" à la base Access
import pyodbc

# pandas affiche un avertissement quand on lui donne une connexion pyodbc (il préfère SQLAlchemy).
# Ça fonctionne très bien quand même, donc on masque ce message pour garder un affichage propre.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

print("Bibliothèques importées.")

Bibliothèques importées.


## 2. Indiquer où se trouve le fichier `.mdb`
Le fichier doit se trouver **dans le même dossier que ce notebook**.

In [2]:
# Path.cwd() = le dossier dans lequel le notebook est ouvert
dossier_projet = Path.cwd()

# On cherche le fichier dont le nom commence par "HuileData2003" et finit par ".mdb"
# (le fichier peut s'appeler "HuileData2003.mdb" ou "HuileData2003(1).mdb")
fichiers_trouves = list(dossier_projet.glob("HuileData2003*.mdb"))

if len(fichiers_trouves) == 0:
    print("ERREUR : aucun fichier HuileData2003*.mdb dans", dossier_projet)
else:
    chemin_base = fichiers_trouves[0]
    print("Fichier trouvé :", chemin_base.name)

Fichier trouvé : HuileData2003(1).mdb


## 3. Se connecter à la base Access
On vérifie d'abord que le **pilote Access** est installé sur l'ordinateur, puis on ouvre la connexion.

> Si le pilote est absent : il faut installer *Microsoft Access Database Engine 2016 Redistributable*
> (version **64 bits** si Python est en 64 bits).

In [3]:
# Nom exact du pilote ODBC fourni par Microsoft pour les fichiers Access
nom_pilote = "Microsoft Access Driver (*.mdb, *.accdb)"

# pyodbc.drivers() donne la liste des pilotes installés sur l'ordinateur
if nom_pilote in pyodbc.drivers():
    print("Pilote Access trouvé.")
else:
    print("ERREUR : pilote Access introuvable. Pilotes disponibles :", pyodbc.drivers())

Pilote Access trouvé.


In [4]:
# La "chaîne de connexion" indique : quel pilote utiliser, quel fichier ouvrir, et en lecture seule
chaine_connexion = (
    "DRIVER={" + nom_pilote + "};"
    + "DBQ=" + str(chemin_base.resolve()) + ";"
    + "ReadOnly=1;"
)

connexion = pyodbc.connect(chaine_connexion)
print("Connexion ouverte (lecture seule).")

Connexion ouverte (lecture seule).


## 4. Voir toutes les tables de la base
Avant de choisir, on regarde **toutes** les tables présentes et leur nombre de lignes.

In [5]:
# Un "curseur" sert à envoyer des commandes à la base
curseur = connexion.cursor()

# On récupère le nom de chaque table (tableType="TABLE" exclut les tables système d'Access)
liste_tables = []
for table in curseur.tables(tableType="TABLE"):
    liste_tables.append(table.table_name)

# Pour chaque table, on compte ses lignes avec la requête SQL "SELECT COUNT(*)"
# Les crochets [ ] autour du nom sont obligatoires car certains noms contiennent des espaces ou des accents
nombre_lignes = []
for nom_table in liste_tables:
    curseur.execute("SELECT COUNT(*) FROM [" + nom_table + "]")
    nombre_lignes.append(curseur.fetchone()[0])

# On met le résultat dans un tableau pandas pour un affichage lisible
inventaire = pd.DataFrame({"table": liste_tables, "nombre_de_lignes": nombre_lignes})
inventaire = inventaire.sort_values("nombre_de_lignes", ascending=False).reset_index(drop=True)

print("La base contient", len(liste_tables), "tables.")
inventaire

La base contient 34 tables.


,table,nombre_de_lignes
0,RésultatsAnalysesCritères,21658
1,Analyse,11703
2,AnalyseDétail,11583
3,Gaz Dissous,5090
4,EPL,3619
5,DBPC,1346
6,BPC,1073
7,Équipement,900
8,Furane,341
9,AnalysesRejetees,254


## 5. Choisir les tables utiles à notre étude
Toutes les tables ne servent pas. On garde celles qui décrivent **les équipements**, **les prélèvements d'huile**,
**les résultats de laboratoire**, **l'avis de l'expert** et **les normes d'interprétation**.

| Table | Ce qu'elle contient | Pourquoi on la garde |
|---|---|---|
| `Équipement` | les 900 équipements : type, site, tension, puissance, fabricant, année | pour isoler les **transformateurs** et les décrire |
| `TypeÉquipement` | traduction du code type (`T` = Transformateur, `1` = Traversée isolée…) | pour lire la table `Équipement` |
| `Localisation` | nom complet des sites et postes (`CIM` = Centrale Isle-Maligne…) | pour lire la table `Équipement` |
| `Analyse` | un enregistrement par **prélèvement d'huile** : équipement, date, type d'essai | c'est le lien entre un équipement et ses résultats |
| `TypeAnalyse` | traduction du code d'essai (`GD` = Gaz dissous, `EPL`…) | pour lire la table `Analyse` |
| `Gaz Dissous` | concentration des gaz dans l'huile (H2, CH4, C2H2, C2H4, C2H6, CO, CO2…) | **mesure principale** pour détecter les défauts |
| `EPL` | qualité de l'huile : rigidité diélectrique, eau, acidité, tension interfaciale, couleur | état de **vieillissement de l'huile** |
| `Furane` | composés issus de la dégradation du papier isolant | état de **vieillissement du papier** |
| `DBPC` | quantité d'antioxydant restant dans l'huile | suivi de l'**oxydation** de l'huile |
| `AnalyseDétail` | pour chaque prélèvement : le code de la recommandation de l'expert | partie **recommandation** du devoir |
| `Recommandation` | texte de chaque recommandation (« Aucune action », « Mettre hors service »…) | pour lire `AnalyseDétail` |
| `NormesC57104` | seuils de la norme IEEE C57.104 pour les gaz dissous | partie **interprétation** du devoir |

Les autres tables sont **écartées** : tables vides (`ECL`, `EP`, `ESL`, `Hydran`), tables de l'application
(utilisateurs, mots de passe, droits d'accès), tables très petites ou hors sujet (`BPC`, `Métaux Dans Huiles`),
ou doublons d'information (normes internes, textes de diagnostic).

In [6]:
# Liste des tables que l'on va charger (les noms doivent être écrits exactement comme dans la base)
tables_a_charger = [
    "Équipement",
    "TypeÉquipement",
    "Localisation",
    "Analyse",
    "TypeAnalyse",
    "Gaz Dissous",
    "EPL",
    "Furane",
    "DBPC",
    "AnalyseDétail",
    "Recommandation",
    "NormesC57104",
]
print(len(tables_a_charger), "tables sélectionnées.")

12 tables sélectionnées.


## 6. Charger les tables dans Python
Pour chaque table, on envoie la requête SQL `SELECT * FROM [nom_table]` (« donne-moi toutes les lignes et toutes les colonnes »)
et `pd.read_sql` range le résultat dans un tableau pandas.

Tous les tableaux sont rangés dans un **dictionnaire** appelé `donnees` :
on retrouve une table en écrivant `donnees["Nom de la table"]`.

In [7]:
donnees = {}   # dictionnaire vide : il va recevoir une table par entrée

for nom_table in tables_a_charger:
    requete = "SELECT * FROM [" + nom_table + "]"
    donnees[nom_table] = pd.read_sql(requete, connexion)
    # .shape donne (nombre de lignes, nombre de colonnes)
    lignes, colonnes = donnees[nom_table].shape
    print(f"{nom_table:15s} : {lignes:6d} lignes, {colonnes:2d} colonnes")

Équipement      :    900 lignes, 12 colonnes
TypeÉquipement  :      5 lignes,  2 colonnes
Localisation    :     23 lignes,  3 colonnes
Analyse         :  11703 lignes, 20 colonnes
TypeAnalyse     :     10 lignes,  4 colonnes
Gaz Dissous     :   5090 lignes, 13 colonnes
EPL             :   3619 lignes, 11 colonnes
Furane          :    341 lignes,  7 colonnes
DBPC            :   1346 lignes,  3 colonnes
AnalyseDétail   :  11583 lignes, 11 colonnes
Recommandation  :     59 lignes,  3 colonnes
NormesC57104    :      3 lignes,  9 colonnes


## 7. Jeter un œil au contenu
`.head()` affiche les 5 premières lignes d'un tableau. On regarde les trois tables les plus importantes.

In [8]:
# Les équipements
donnees["Équipement"].head()

,NoÉquipement,TypeÉquipement,Site,Localisation,LitreHuile,NomÉquipement,NoSérie,Tension,Puissance,Manufacturier,Année,Modifier
0,10000,1,CCP,CCP,0.0,Traverse isolée,10000,0.0,0.0,,0.0,False
1,10002,1,CCP,CCP,270.0,Traversée isolée H1,238699,362.0,1600.0,ABB,0.0,False
2,10008,1,CCP,CCP,273.0,Traverse Isolée H1,10008,330.0,600.0,WEST,0.0,False
3,10017,1,CCP,CCP,273.0,Traverse Isolée H1,10017,330.0,600.0,WEST,0.0,False
4,10018,1,CCP,CCP,0.0,Traversée isolée H0,3S5933F,25.0,800.0,WEST,0.0,False


In [9]:
# Les prélèvements d'huile (une ligne = un échantillon envoyé au laboratoire)
donnees["Analyse"].head()

,CléAnalyse,NoÉquipement,DateAnalyse,TypeAnalyse,NoAnalyse,NoExploitation,CodeMatière,CodeMotif,CodeLieu,CodeLaboratoire,Payé,NoMSO,TypeHuile,Charge,DatePrélèvement,Remarque,PrélevéPar,Modifier,Transmission,Impression
0,AM0000000000001,11001,1986-10-29,EPL,,B13-B-T,0,1.0,1,0,True,None,0,0.0,1986-09-17,,,False,False,False
1,AM0000000000002,11002,1986-10-29,EPL,,B13-C-T,0,1.0,1,0,True,None,0,0.0,1986-09-17,,,False,False,False
2,AM0000000000003,11003,1986-10-29,EPL,,B14-A-T,0,1.0,1,0,True,None,0,0.0,1986-09-17,,,False,False,False
3,AM0000000000004,11004,1986-10-29,EPL,,B14-B-T,0,1.0,1,0,True,None,0,0.0,1986-09-17,,,False,False,False
4,AM0000000000005,11005,1986-10-29,EPL,,B14-C-T,0,1.0,1,0,True,None,0,0.0,1986-09-17,,,False,False,False


In [10]:
# Les résultats de gaz dissous (en ppm = parties par million)
donnees["Gaz Dissous"].head()

,CléAnalyse,NoÉquipement,H2,CH4,C2H2,C2H4,C2H6,CO,CO2,O2,N2,MWs,Température
0,AM0000000018012,12059,155.0,45.0,1.0,4.0,13.0,275.0,1634.0,9068.0,79531.0,0.0,0.0
1,AM0000000018014,12061,20.0,10.0,2.0,3.0,2.0,468.0,1270.0,1920.0,96400.0,0.0,0.0
2,AM0000000018016,12063,20.0,10.0,2.0,2.0,12.0,209.0,416.0,1970.0,121000.0,0.0,0.0
3,AM0000000018022,12072,30.0,15.0,0.0,7.0,23.0,558.0,2160.0,9070.0,81000.0,0.0,0.0
4,AM0000000018023,12073,25.0,10.0,0.0,0.0,14.0,72.0,803.0,8880.0,89200.0,0.0,0.0


## 8. Combien de transformateurs dans la base ?
La table `Équipement` contient 900 équipements, mais **pas seulement des transformateurs**.
On compte les équipements par type, en traduisant le code grâce à la table `TypeÉquipement`.

In [11]:
equipements = donnees["Équipement"]
types = donnees["TypeÉquipement"]

# merge = on "colle" à chaque équipement le libellé de son type (comme une RECHERCHEV dans Excel)
equipements_avec_libelle = equipements.merge(types, on="TypeÉquipement", how="left")

# value_counts() compte combien de fois chaque libellé apparaît
equipements_avec_libelle["DescriptionTypeÉquipement"].value_counts()

DescriptionTypeÉquipement
Traverse isolée    439
Transformateur     432
Disjoncteur         24
Réservoir            5
Name: count, dtype: int64

**Lecture :** sur les 900 équipements, **432 sont des transformateurs**. Les autres sont surtout des traversées isolées
(bornes de raccordement en porcelaine, elles aussi remplies d'huile), quelques disjoncteurs et réservoirs.
Comme notre thème porte sur les **transformateurs de puissance**, la suite de l'étude se concentrera sur ces 432 équipements
(le filtre sera appliqué à l'étape de préparation).

## 9. Sauvegarder une copie CSV de chaque table
Les fichiers sont enregistrés dans `data/raw/` (« raw » = données brutes, non modifiées).
On enlève les accents et les espaces des noms de fichiers pour éviter les problèmes entre ordinateurs.

In [12]:
dossier_raw = dossier_projet / "data" / "raw"
dossier_raw.mkdir(parents=True, exist_ok=True)   # crée le dossier s'il n'existe pas


def nom_sans_accent(texte):
    # "Gaz Dissous" -> "Gaz_Dissous" ; "Équipement" -> "Equipement"
    sans_accent = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return sans_accent.replace(" ", "_")


for nom_table, tableau in donnees.items():
    fichier = dossier_raw / (nom_sans_accent(nom_table) + ".csv")
    tableau.to_csv(fichier, index=False, encoding="utf-8")   # index=False : on n'écrit pas les numéros de ligne
    print("Enregistré :", fichier.relative_to(dossier_projet))

Enregistré : data\raw\Equipement.csv
Enregistré : data\raw\TypeEquipement.csv
Enregistré : data\raw\Localisation.csv
Enregistré : data\raw\Analyse.csv
Enregistré : data\raw\TypeAnalyse.csv
Enregistré : data\raw\Gaz_Dissous.csv
Enregistré : data\raw\EPL.csv
Enregistré : data\raw\Furane.csv
Enregistré : data\raw\DBPC.csv
Enregistré : data\raw\AnalyseDetail.csv
Enregistré : data\raw\Recommandation.csv
Enregistré : data\raw\NormesC57104.csv


In [13]:
# On ferme la connexion : toutes les données sont maintenant en mémoire et dans les CSV
connexion.close()
print("Connexion fermée.")

Connexion fermée.


## Bilan de l'étape
- La base contient **34 tables** ; nous en avons chargé **12**, celles qui servent notre étude.
- Les données sont dans le dictionnaire `donnees` et sauvegardées en CSV dans `data/raw/`.
- La base compte **900 équipements, dont 432 transformateurs** : ce sont eux que nous étudierons.

**Prochaine étape :** explorer la qualité de ces données (valeurs manquantes, erreurs de saisie) avant de les préparer.